In [1]:
import re
import json
from typing import List, Dict, Tuple

# 1. 静态医学同义词词典（按要求定义，可扩展UMLS/MeSH）
MEDICAL_SYNONYMS = {
    "mi": ["myocardial infarction", "heart attack"],
    "dm": ["diabetes mellitus", "type 2 diabetes", "diabetes"],
    "htn": ["hypertension", "high blood pressure"],
    "cad": ["coronary artery disease", "coronary heart disease"],
    "asa": ["aspirin", "acetylsalicylic acid"],
    "metformin": ["glucophage", "dimethylbiguanide"],
    "statin": ["atorvastatin", "rosuvastatin", "simvastatin"],
    "warfarin": ["coumadin"],
    "insulin": ["human insulin", "recombinant insulin"]
}

# 2. 修正后的医学实体正则匹配规则（适配中英文混合场景）
# 用 (?<!\w) / (?!\w) 替代 \b，解决中文语境下单词边界失效问题
MEDICAL_PATTERNS = {
    "drug": r"(?<![a-zA-Z])(aspirin|metformin|atorvastatin|rosuvastatin|simvastatin|warfarin|insulin|glucophage|coumadin)(?![a-zA-Z])",
    "disease": r"(?<![a-zA-Z])(myocardial infarction|heart attack|diabetes|hypertension|coronary artery disease)(?![a-zA-Z])",
    "abbreviation": r"(?<![a-zA-Z])(mi|dm|htn|cad|asa)(?![a-zA-Z])"
}

# 3. 核心工具函数（全量修正，保证逻辑闭环）
def basic_clean(query: str) -> str:
    """基础清洗：去除多余空格、特殊字符，统一格式"""
    query = re.sub(r"\s+", " ", query).strip()
    query = re.sub(r"[^\w\s\?\.,()]", "", query)
    return query

def recognize_medical_entities(query: str) -> Dict[str, List[str]]:
    """识别医学实体：适配中英文混合场景，解决\b边界失效问题"""
    entities = {}
    for entity_type, pattern in MEDICAL_PATTERNS.items():
        matches = re.findall(pattern, query, re.IGNORECASE)
        # 去重、统一小写
        entities[entity_type] = list(set([match.lower() for match in matches]))
    return entities

def expand_medical_synonyms(query: str, entities: Dict[str, List[str]]) -> str:
    """同义词扩展：替换医学缩写/术语为完整表达，优先长词避免冲突"""
    expanded_query = query
    # 遍历所有实体，优先替换长词避免部分匹配
    for term in sorted(MEDICAL_SYNONYMS.keys(), key=len, reverse=True):
        term_lower = term.lower()
        # 检查term是否在识别到的实体中（覆盖缩写/药物/疾病）
        for entity_list in entities.values():
            if term_lower in [e.lower() for e in entity_list]:
                # 完整匹配替换，不影响上下文
                expanded_query = re.sub(
                    rf"(?<!\w){re.escape(term_lower)}(?!\w)",
                    MEDICAL_SYNONYMS[term][0],
                    expanded_query,
                    flags=re.IGNORECASE
                )
                break
    return expanded_query

def extract_filter_conditions(query: str) -> Dict:
    """从查询中提取过滤条件（时间范围等，可扩展）"""
    filters = {}
    # 示例：提取年份（可根据业务扩展）
    time_match = re.search(r"\b(20\d{2}|19\d{2})\b", query)
    if time_match:
        filters["year"] = time_match.group(1)
    return filters

# 4. 生成多版本查询（严格对齐BGE最佳实践）
def generate_vector_query(enhanced_query: str) -> str:
    """BGE模型最佳实践：添加指令前缀，用于向量检索"""
    return f"Represent this question for searching relevant passages: {enhanced_query}"

def generate_keyword_query(entities: Dict[str, List[str]], expanded_query: str) -> str:
    """生成关键词查询：用于关键词检索/过滤，去重拼接"""
    keywords = []
    # 提取所有实体作为关键词
    for entity_list in entities.values():
        keywords.extend(entity_list)
    # 去重、拼接
    keyword_str = " OR ".join(list(set(keywords)))
    return f"({keyword_str}) AND {expanded_query}"

# 5. 主函数：完整查询理解与增强流程
def process_medical_query(original_query: str) -> Tuple[str, str, Dict, Dict]:
    """
    完整处理医学查询，返回增强后的多版本查询、实体、过滤条件
    返回:
        vector_query: 用于向量检索的增强查询（BGE格式）
        keyword_query: 用于关键词检索的增强查询
        entities: 识别到的医学实体
        filters: 提取的过滤条件
    """
    # 1. 基础清洗
    cleaned_query = basic_clean(original_query)
    
    # 2. 识别医学实体
    entities = recognize_medical_entities(cleaned_query)
    
    # 3. 同义词扩展
    expanded_query = expand_medical_synonyms(cleaned_query, entities)
    
    # 4. 提取过滤条件
    filters = extract_filter_conditions(expanded_query)
    
    # 5. 生成向量查询（BGE指令前缀）
    vector_query = generate_vector_query(expanded_query)
    
    # 6. 生成关键词查询
    keyword_query = generate_keyword_query(entities, expanded_query)
    
    return vector_query, keyword_query, entities, filters

# 6. 示例调用（验证功能，直接运行看结果）
if __name__ == "__main__":
    # 测试用例1：含医学缩写的查询
    test_query_1 = "mi对心血管疾病有何影响？"
    print("="*80)
    print("测试用例1：", test_query_1)
    vq1, kq1, ent1, filt1 = process_medical_query(test_query_1)
    print("\n【向量查询（BGE格式）】：", vq1)
    print("\n【关键词查询】：", kq1)
    print("\n【识别到的医学实体】：", json.dumps(ent1, indent=2, ensure_ascii=False))
    print("\n【提取的过滤条件】：", json.dumps(filt1, indent=2, ensure_ascii=False))
    
    # 测试用例2：含药物术语的查询
    test_query_2 = "二甲双胍(metformin)对2型糖尿病患者的心血管获益是什么？"
    print("\n" + "="*80)
    print("测试用例2：", test_query_2)
    vq2, kq2, ent2, filt2 = process_medical_query(test_query_2)
    print("\n【向量查询（BGE格式）】：", vq2)
    print("\n【关键词查询】：", kq2)
    print("\n【识别到的医学实体】：", json.dumps(ent2, indent=2, ensure_ascii=False))
    print("\n【提取的过滤条件】：", json.dumps(filt2, indent=2, ensure_ascii=False))

测试用例1： mi对心血管疾病有何影响？

【向量查询（BGE格式）】： Represent this question for searching relevant passages: mi对心血管疾病有何影响

【关键词查询】： (mi) AND mi对心血管疾病有何影响

【识别到的医学实体】： {
  "drug": [],
  "disease": [],
  "abbreviation": [
    "mi"
  ]
}

【提取的过滤条件】： {}

测试用例2： 二甲双胍(metformin)对2型糖尿病患者的心血管获益是什么？

【向量查询（BGE格式）】： Represent this question for searching relevant passages: 二甲双胍(glucophage)对2型糖尿病患者的心血管获益是什么

【关键词查询】： (metformin) AND 二甲双胍(glucophage)对2型糖尿病患者的心血管获益是什么

【识别到的医学实体】： {
  "drug": [
    "metformin"
  ],
  "disease": [],
  "abbreviation": []
}

【提取的过滤条件】： {}


In [2]:
import os
# 全局强制 Hugging Face 库完全离线，永不联网
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import re
import math
import chromadb
import json
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Transformer, Pooling
from transformers import AutoModel, AutoTokenizer
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

# 1. 复用之前的查询理解与增强核心函数
# （保持和之前完全一致，确保无缝衔接）
MEDICAL_SYNONYMS = {
    "mi": ["myocardial infarction", "heart attack"],
    "dm": ["diabetes mellitus", "type 2 diabetes", "diabetes"],
    "htn": ["hypertension", "high blood pressure"],
    "cad": ["coronary artery disease", "coronary heart disease"],
    "asa": ["aspirin", "acetylsalicylic acid"],
    "metformin": ["glucophage", "dimethylbiguanide"],
    "statin": ["atorvastatin", "rosuvastatin", "simvastatin"],
    "warfarin": ["coumadin"],
    "insulin": ["human insulin", "recombinant insulin"]
}

MEDICAL_PATTERNS = {
    "drug": r"(?<![a-zA-Z])(aspirin|metformin|atorvastatin|rosuvastatin|simvastatin|warfarin|insulin|glucophage|coumadin)(?![a-zA-Z])",
    "disease": r"(?<![a-zA-Z])(myocardial infarction|heart attack|diabetes|hypertension|coronary artery disease)(?![a-zA-Z])",
    "abbreviation": r"(?<![a-zA-Z])(mi|dm|htn|cad|asa)(?![a-zA-Z])"
}

def basic_clean(query: str) -> str:
    query = re.sub(r"\s+", " ", query).strip()
    query = re.sub(r"[^\w\s\?\.,()]", "", query)
    return query

def recognize_medical_entities(query: str) -> Dict[str, List[str]]:
    entities = {}
    for entity_type, pattern in MEDICAL_PATTERNS.items():
        matches = re.findall(pattern, query, re.IGNORECASE)
        entities[entity_type] = list(set([match.lower() for match in matches]))
    return entities

def expand_medical_synonyms(query: str, entities: Dict[str, List[str]]) -> str:
    expanded_query = query
    for term in sorted(MEDICAL_SYNONYMS.keys(), key=len, reverse=True):
        term_lower = term.lower()
        for entity_list in entities.values():
            if term_lower in [e.lower() for e in entity_list]:
                expanded_query = re.sub(
                    rf"(?<!\w){re.escape(term_lower)}(?!\w)",
                    MEDICAL_SYNONYMS[term][0],
                    expanded_query,
                    flags=re.IGNORECASE
                )
                break
    return expanded_query

def process_medical_query(original_query: str) -> Tuple[str, Dict]:
    """处理用户查询，返回增强后的查询和实体信息（对应query_info）"""
    cleaned = basic_clean(original_query)
    entities = recognize_medical_entities(cleaned)
    expanded = expand_medical_synonyms(cleaned, entities)
    return expanded, entities

# 2. BM25 核心实现
class BM25:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.documents: List[List[str]] = []  # 分词后的文档
        self.avgdl: float = 0.0
        self.df: Dict[str, int] = defaultdict(int)  # 文档频率
        self.idf: Dict[str, float] = {}  # 逆文档频率
        self.doc_len: List[int] = []  # 每个文档的长度

    def _tokenize(self, text: str) -> List[str]:
        """简单分词（英文按空格/标点分割，适配医学术语；中文可替换为jieba）"""
        text = re.sub(r"[^\w\s]", " ", text.lower())
        return [token for token in text.split() if token.strip()]

    def fit(self, documents: List[str]) -> None:
        """构建BM25索引：输入原始文档列表，计算idf、avgdl等"""
        self.documents = [self._tokenize(doc) for doc in documents]
        self.doc_len = [len(doc) for doc in self.documents]
        self.avgdl = sum(self.doc_len) / len(self.doc_len) if self.doc_len else 0.0

        # 计算文档频率df
        for doc in self.documents:
            unique_tokens = set(doc)
            for token in unique_tokens:
                self.df[token] += 1

        # 计算逆文档频率idf
        n_docs = len(self.documents)
        for token, df in self.df.items():
            self.idf[token] = math.log((n_docs - df + 0.5) / (df + 0.5) + 1)

    def get_scores(self, query_tokens: List[str]) -> List[float]:
        """计算查询与所有文档的BM25分数"""
        scores = []
        for doc_idx, doc in enumerate(self.documents):
            score = 0.0
            doc_len = self.doc_len[doc_idx]
            # 统计词频tf
            tf = defaultdict(int)
            for token in doc:
                tf[token] += 1
            # 计算BM25分数
            for token in query_tokens:
                if token not in self.idf:
                    continue
                idf = self.idf[token]
                tf_val = tf.get(token, 0)
                numerator = idf * tf_val * (self.k1 + 1)
                denominator = tf_val + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                score += numerator / denominator
            scores.append(score)
        return scores

    def search(self, query: str, top_k: int = 5) -> List[Tuple[int, float]]:
        """执行BM25检索，返回(文档索引, 分数)的top_k结果"""
        query_tokens = self._tokenize(query)
        scores = self.get_scores(query_tokens)
        # 按分数降序排序，取top_k
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

# 3. 多路检索（MultiPathRetriever）核心类
class MultiPathRetriever:
    def __init__(
        self,
        chroma_collection: chromadb.Collection,
        embedding_model: SentenceTransformer,
        bm25: BM25,
        documents: List[str],
        metadatas: Optional[List[Dict]] = None
    ):
        """
        初始化多路检索器
        :param chroma_collection: ChromaDB的集合（向量检索用）
        :param embedding_model: BGE嵌入模型
        :param bm25: 已训练好的BM25实例
        :param documents: 原始文档列表（和ChromaDB、BM25一一对应）
        :param metadatas: 文档元数据列表（可选，和documents一一对应）
        """
        self.chroma_collection = chroma_collection
        self.embedding_model = embedding_model
        self.bm25 = bm25
        self.documents = documents
        self.metadatas = metadatas if metadatas is not None else [{} for _ in documents]

    def _vector_search(self, query: str, top_k: int) -> List[Dict]:
        """向量检索：ChromaDB语义检索"""
        # 生成BGE格式查询
        bge_query = f"Represent this question for searching relevant passages: {query}"
        query_embedding = self.embedding_model.encode(bge_query).tolist()
        results = self.chroma_collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )
        # 格式化结果：{doc_id, content, metadata, score, source}
        vector_results = []
        for idx in range(len(results["documents"][0])):
            doc_id = results["ids"][0][idx] if "ids" in results else f"vec_{idx}"
            vector_results.append({
                "doc_id": doc_id,
                "content": results["documents"][0][idx],
                "metadata": results["metadatas"][0][idx] if results["metadatas"] else {},
                "score": 1 - results["distances"][0][idx],  # 距离转相似度
                "source": "vector"
            })
        return vector_results

    def _keyword_search(self, query: str, top_k: int) -> List[Dict]:
        """关键词检索：BM25检索"""
        bm25_results = self.bm25.search(query, top_k=top_k)
        # 格式化结果
        keyword_results = []
        for doc_idx, score in bm25_results:
            keyword_results.append({
                "doc_id": f"kw_{doc_idx}",
                "content": self.documents[doc_idx],
                "metadata": self.metadatas[doc_idx],
                "score": score,
                "source": "keyword"
            })
        return keyword_results

    def _fusion_results(
        self,
        vector_results: List[Dict],
        keyword_results: List[Dict],
        strategy: str = "rrf",
        vector_weight: float = 0.7
    ) -> List[Dict]:
        """
        融合两路检索结果
        :param strategy: 融合策略 ('rrf', 'weighted', 'simple')
        :param vector_weight: 加权融合时向量检索的权重（keyword权重为1-vector_weight）
        """
        # 1. 简单合并去重（simple）
        if strategy == "simple":
            merged = {}
            # 先合并向量结果
            for res in vector_results:
                merged[res["content"]] = res
            # 再合并关键词结果，去重
            for res in keyword_results:
                if res["content"] not in merged:
                    merged[res["content"]] = res
            return list(merged.values())

        # 2. RRF融合（Reciprocal Rank Fusion）
        elif strategy == "rrf":
            rrf_const = 60  # RRF默认常数
            doc_scores = defaultdict(float)
            doc_map = {}

            # 处理向量检索排名
            for rank, res in enumerate(vector_results, 1):
                key = res["content"]
                doc_map[key] = res
                doc_scores[key] += 1 / (rank + rrf_const)

            # 处理关键词检索排名
            for rank, res in enumerate(keyword_results, 1):
                key = res["content"]
                if key not in doc_map:
                    doc_map[key] = res
                doc_scores[key] += 1 / (rank + rrf_const)

            # 按RRF分数排序
            sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
            return [doc_map[key] for key, _ in sorted_docs]

        # 3. 加权融合（向量权重更高）
        elif strategy == "weighted":
            keyword_weight = 1 - vector_weight
            doc_scores = defaultdict(float)
            doc_map = {}

            # 归一化分数（避免量纲差异）
            def _normalize_scores(results: List[Dict]) -> List[Dict]:
                if not results:
                    return []
                max_score = max(res["score"] for res in results)
                min_score = min(res["score"] for res in results)
                if max_score == min_score:
                    for res in results:
                        res["norm_score"] = 1.0
                else:
                    for res in results:
                        res["norm_score"] = (res["score"] - min_score) / (max_score - min_score)
                return results

            vector_results = _normalize_scores(vector_results)
            keyword_results = _normalize_scores(keyword_results)

            # 计算加权分数
            for res in vector_results:
                key = res["content"]
                doc_map[key] = res
                doc_scores[key] += res["norm_score"] * vector_weight

            for res in keyword_results:
                key = res["content"]
                if key not in doc_map:
                    doc_map[key] = res
                doc_scores[key] += res["norm_score"] * keyword_weight

            # 按加权分数排序
            sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
            return [doc_map[key] for key, _ in sorted_docs]

        else:
            raise ValueError(f"不支持的融合策略: {strategy}，可选: rrf, weighted, simple")

    def retrieve(
        self,
        query_info: Tuple[str, Dict],
        top_k_vector: int = 5,
        top_k_keyword: int = 5,
        fusion_strategy: str = "rrf"
    ) -> List[Dict]:
        """
        多路检索主入口（完全符合需求的Args要求）
        :param query_info: 查询处理器输出的信息 -> (增强后的查询, 实体信息)
        :param top_k_vector: 向量检索返回数量
        :param top_k_keyword: 关键词检索返回数量
        :param fusion_strategy: 融合策略 ('rrf', 'weighted', 'simple')
        :return: 融合后的最终检索结果
        """
        expanded_query, entities = query_info

        # 1. 执行两路检索
        vector_results = self._vector_search(expanded_query, top_k_vector)
        keyword_results = self._keyword_search(expanded_query, top_k_keyword)

        # 2. 融合结果
        final_results = self._fusion_results(
            vector_results,
            keyword_results,
            strategy=fusion_strategy
        )

        # 3. 补充实体信息，生成格式化结果
        for res in final_results:
            res["entities"] = entities

        return final_results

# 4. 示例调用（可直接运行）
if __name__ == "__main__":
    # 模拟初始化（实际替换为你的真实数据）
    # 1. 模拟医学文档库（实际替换为你的真实医学文献）
    sample_docs = [
        "Myocardial infarction (MI) is a major risk factor for cardiovascular disease, increasing mortality risk.",
        "Metformin (glucophage) improves cardiovascular outcomes in type 2 diabetes patients by reducing blood glucose.",
        "MI patients have higher incidence of heart failure and coronary artery disease.",
        "Type 2 diabetes patients on metformin show lower cardiovascular events compared to placebo.",
        "Hypertension (HTN) is a key risk factor for myocardial infarction and stroke."
    ]
    sample_metadatas = [
        {"source": "Cardiology Journal 2024", "topic": "MI & CVD"},
        {"source": "Diabetes Care 2023", "topic": "Metformin & CVD"},
        {"source": "Circulation 2024", "topic": "MI Complications"},
        {"source": "NEJM 2023", "topic": "Metformin Trials"},
        {"source": "Hypertension 2024", "topic": "HTN & MI"}
    ]

    # 2. 初始化ChromaDB（实际替换为你已有的向量库）
    client = chromadb.PersistentClient(path="./chroma_db")
    # 若集合不存在，创建并插入数据（实际用你已有的集合）
    try:
        collection = client.get_collection(name="medical_collection")
    except:
        collection = client.create_collection(name="medical_collection")
        collection.add(
            documents=sample_docs,
            metadatas=sample_metadatas,
            ids=[f"doc_{i}" for i in range(len(sample_docs))]
        )

    # 3. 加载BGE嵌入模型
    # 1. 定义模型本地路径
    model_dir = "/root/.cache/huggingface/hub/models--BAAI--bge-small-en-v1.5/snapshots/latest/"
    # 4. 【核心步骤】手动组装 SentenceTransformer 流水线（完全绕开校验）
    # 4.1 加载原生 Transformer 模块（用本地路径，不触发校验）
    transformer_module = Transformer(model_dir, model_args={"local_files_only": True})
    # 4.2 加载 Pooling 模块（BGE 模型必须用 mean pooling）
    pooling_dim = transformer_module.get_word_embedding_dimension()
    pooling_module = Pooling(pooling_dim, pooling_mode_mean_tokens=True)
    # 4.3 手动组装成最终模型（100% 绕过 SentenceTransformer 内部的 load 逻辑）
    embedding_model = SentenceTransformer(modules=[transformer_module, pooling_module])
    # 5. 测试一下（必做，确认模型正常）
    print("模型加载成功！向量维度测试：", embedding_model.encode("测试").shape)

    # 4. 初始化并训练BM25
    bm25 = BM25()
    bm25.fit(sample_docs)

    # 5. 初始化多路检索器
    retriever = MultiPathRetriever(
        chroma_collection=collection,
        embedding_model=embedding_model,
        bm25=bm25,
        documents=sample_docs,
        metadatas=sample_metadatas
    )

    # 测试多路检索（完全符合需求）
    # 测试用例1
    print("="*80)
    print("测试用例1：mi对心血管疾病有何影响？")
    query_info_1 = process_medical_query("mi对心血管疾病有何影响？")
    results_1 = retriever.retrieve(
        query_info=query_info_1,
        top_k_vector=3,
        top_k_keyword=3,
        fusion_strategy="rrf"
    )
    print(f"\n【融合后结果（RRF策略）】：")
    for idx, res in enumerate(results_1, 1):
        print(f"\n📌 结果 {idx}")
        print(f"来源：{res['source']}")
        print(f"内容：{res['content']}")
        print(f"元数据：{res['metadata']}")
        print(f"识别实体：{res['entities']}")

    # 测试用例2
    print("\n" + "="*80)
    print("测试用例2：二甲双胍(metformin)对2型糖尿病患者的心血管获益是什么？")
    query_info_2 = process_medical_query("二甲双胍(metformin)对2型糖尿病患者的心血管获益是什么？")
    results_2 = retriever.retrieve(
        query_info=query_info_2,
        top_k_vector=3,
        top_k_keyword=3,
        fusion_strategy="weighted"
    )
    print(f"\n【融合后结果（加权策略，向量权重0.7）】：")
    for idx, res in enumerate(results_2, 1):
        print(f"\n📌 结果 {idx}")
        print(f"来源：{res['source']}")
        print(f"内容：{res['content']}")
        print(f"元数据：{res['metadata']}")
        print(f"识别实体：{res['entities']}")

/root/miniconda3/envs/med_rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_1410/2282229655.py:11: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers.models import Transformer, Pooling
The Transformer `model_args` argument was renamed and is now deprecated, please use `model_kwargs` instead.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20790.78it/s]
/tmp/ipykernel_1410/2282229655.py:357: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling_dim = transformer_module.get_word_embedding_dimension()
/root/miniconda3/envs/med_rag/lib/python3.10/site-pac

模型加载成功！向量维度测试： (384,)
测试用例1：mi对心血管疾病有何影响？

【融合后结果（RRF策略）】：

📌 结果 1
来源：vector
内容：Myocardial infarction (MI) is a major risk factor for cardiovascular disease, increasing mortality risk.
元数据：{'source': 'Cardiology Journal 2024', 'topic': 'MI & CVD'}
识别实体：{'drug': [], 'disease': [], 'abbreviation': ['mi']}

📌 结果 2
来源：vector
内容：MI patients have higher incidence of heart failure and coronary artery disease.
元数据：{'source': 'Circulation 2024', 'topic': 'MI Complications'}
识别实体：{'drug': [], 'disease': [], 'abbreviation': ['mi']}

📌 结果 3
来源：vector
内容：Hypertension (HTN) is a key risk factor for myocardial infarction and stroke.
元数据：{'source': 'Hypertension 2024', 'topic': 'HTN & MI'}
识别实体：{'drug': [], 'disease': [], 'abbreviation': ['mi']}

📌 结果 4
来源：keyword
内容：Metformin (glucophage) improves cardiovascular outcomes in type 2 diabetes patients by reducing blood glucose.
元数据：{'source': 'Diabetes Care 2023', 'topic': 'Metformin & CVD'}
识别实体：{'drug': [], 'disease': [], 'abbreviation': ['mi']}

测试用例

In [3]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import torch
import re
import json
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Transformer, Pooling
from sklearn.metrics.pairwise import cosine_similarity

# 2. 最终版重排序器（手动组装，零报错）
class BGEReranker:
    def __init__(self, embed_model_dir: str):
        self.embed_model_dir = embed_model_dir
        
        # 🔴 手动加载 Transformer 和 Pooling 模块，组装 SentenceTransformer（你要的这段完整保留）
        transformer = Transformer(embed_model_dir)
        pooling = Pooling(transformer.get_word_embedding_dimension())
        self.model = SentenceTransformer(modules=[transformer, pooling])
        self.model.eval()

        # 多准则权重配置
        self.criteria_weights = {
            "relevance": 0.6,
            "recency": 0.25,
            "authority": 0.15
        }
        self.journal_authority = {
            "Circulation": 1.0,
            "Cardiology Journal 2024": 0.9,
            "Hypertension 2024": 0.85,
            "Diabetes Care 2023": 0.95
        }

    def _compute_relevance_score(self, query: str, doc_text: str) -> float:
        query_emb = self.model.encode([query], convert_to_tensor=True)
        doc_emb = self.model.encode([doc_text], convert_to_tensor=True)
        similarity = cosine_similarity(query_emb.cpu().numpy(), doc_emb.cpu().numpy())[0][0]
        return float((similarity + 1) / 2)

    def _compute_recency_score(self, meta_data: dict) -> float:
        source = meta_data.get("source", "")
        year_match = re.search(r"\d{4}", source)
        if not year_match:
            return 0.5
        year = int(year_match.group())
        current_year = datetime.now().year
        return max(0.2, 1.0 - (current_year - year) * 0.05)

    def _compute_authority_score(self, meta_data: dict) -> float:
        source = meta_data.get("source", "")
        journal_name = re.sub(r"\s*\d{4}$", "", source).strip()
        return self.journal_authority.get(journal_name, 0.5)

    def rerank(self, query: str, search_results: list) -> list:
        reranked_results = []
        for result in search_results:
            rel_score = self._compute_relevance_score(query, result["content"])
            rec_score = self._compute_recency_score(result.get("meta_data", {}))
            auth_score = self._compute_authority_score(result.get("meta_data", {}))

            final_score = (
                rel_score * self.criteria_weights["relevance"] +
                rec_score * self.criteria_weights["recency"] +
                auth_score * self.criteria_weights["authority"]
            )

            result["final_score"] = final_score
            result["rel_score"] = rel_score
            result["rec_score"] = rec_score
            result["auth_score"] = auth_score
            reranked_results.append(result)

        reranked_results.sort(key=lambda x: x["final_score"], reverse=True)
        return reranked_results

# ===================== 3. 使用示例（直接替换路径即可运行） =====================
if __name__ == "__main__":
    # 👉 替换为你的嵌入模型目录（你已经跑通的那个）
    embed_model_dir = "/root/.cache/huggingface/hub/models--BAAI--bge-small-en-v1.5/snapshots/latest/"

    # 初始化重排序器
    reranker = BGEReranker(embed_model_dir)

    # 模拟你的检索结果
    initial_results = [
        {
            "content": "Myocardial infarction (MI) is a major risk factor for cardiovascular disease, increasing mortality risk.",
            "meta_data": {'source': 'Cardiology Journal 2024', 'topic': 'MI & CVD'}
        },
        {
            "content": "MI patients have higher incidence of heart failure and coronary artery disease.",
            "meta_data": {'topic': 'MI Complications', 'source': 'Circulation 2024'}
        }
    ]

    # 执行重排序
    query = "mi对心血管疾病有何影响？"
    ranked_results = reranker.rerank(query, initial_results)

    # 打印结果
    print("="*50)
    print(f"用户查询：{query}")
    print("="*50)
    for idx, res in enumerate(ranked_results, 1):
        print(f"\n【重排序后 Top{idx}】")
        print(f"综合分数：{res['final_score']:.4f}")
        print(f"相似度分数：{res['rel_score']:.4f}")
        print(f"内容：{res['content']}")
        print(f"来源：{res['meta_data']['source']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20445.49it/s]

用户查询：mi对心血管疾病有何影响？

【重排序后 Top1】
综合分数：0.8498
相似度分数：0.7913
内容：MI patients have higher incidence of heart failure and coronary artery disease.
来源：Circulation 2024

【重排序后 Top2】
综合分数：0.7725
相似度分数：0.7874
内容：Myocardial infarction (MI) is a major risk factor for cardiovascular disease, increasing mortality risk.
来源：Cardiology Journal 2024



/tmp/ipykernel_1410/2849137333.py:19: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pooling = Pooling(transformer.get_word_embedding_dimension())


In [4]:
# 1. 依赖导入（和之前的代码放一起也不冲突）
from dataclasses import dataclass
from typing import Dict, Any, List, Set
import re
from transformers import AutoTokenizer

# 2. 定义文档块数据类
@dataclass
class DocumentChunk:
    text: str
    metadata: Dict[str, Any]
    relevance_score: float
    source: str
    chunk_id: str

# 3. 上下文组装器
class ContextAssembler:
    def __init__(self, tokenizer_name_or_path: str = "bert-base-uncased", max_context_tokens: int = 2048):
        # 加载tokenizer（和你之前的嵌入/reranker环境兼容，支持本地路径）
        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_name_or_path, 
            local_files_only=True  # 适配你无网环境
        )
        self.max_context_tokens = max_context_tokens

    def estimate_tokens(self, text: str) -> int:
        """估算文本token数量"""
        return len(self.tokenizer.encode(text, add_special_tokens=False))

    def jaccard_similarity(self, text1: str, text2: str) -> float:
        """计算文本Jaccard相似度，用于去重"""
        words1 = set(re.findall(r"\w+", text1.lower()))
        words2 = set(re.findall(r"\w+", text2.lower()))
        if not words1 or not words2:
            return 0.0
        intersection = len(words1 & words2)
        union = len(words1 | words2)
        return intersection / union

    def deduplicate_chunks(self, chunks: List[DocumentChunk], threshold: float = 0.8) -> List[DocumentChunk]:
        """文本去重：Jaccard相似度超过阈值则视为重复"""
        unique_chunks: List[DocumentChunk] = []
        seen_texts: Set[str] = set()

        for chunk in chunks:
            # 简单预处理文本，用于相似度判断
            clean_text = re.sub(r"\s+", " ", chunk.text.strip())
            is_duplicate = False

            # 和已保留的chunk做相似度判断
            for seen_chunk in unique_chunks:
                sim = self.jaccard_similarity(clean_text, seen_chunk.text)
                if sim >= threshold:
                    is_duplicate = True
                    break

            if not is_duplicate:
                unique_chunks.append(chunk)
                seen_texts.add(clean_text)

        return unique_chunks

    def prioritize_chunks(self, chunks: List[DocumentChunk], max_per_source: int = 3) -> List[DocumentChunk]:
        """优先级排序：按相关性降序，同时限制同一来源文档数量，保证多样性"""
        # 1. 先按相关性降序排序
        sorted_chunks = sorted(chunks, key=lambda x: x.relevance_score, reverse=True)

        # 2. 限制同一来源的文档数量，保证多样性
        source_count: Dict[str, int] = {}
        prioritized = []

        for chunk in sorted_chunks:
            source = chunk.source
            count = source_count.get(source, 0)

            # 超过限制的文档，暂时放到备选池，优先保留不同来源的高相关文档
            if count < max_per_source:
                prioritized.append(chunk)
                source_count[source] = count + 1

        return prioritized

    def truncate_context(self, chunks: List[DocumentChunk]) -> str:
        """构建上下文并按token限制截断，在完整句子处结束"""
        context_parts = []
        total_tokens = 0
        # 预留部分token给特殊符号和后续拼接
        max_available_tokens = self.max_context_tokens - 100

        for chunk in chunks:
            chunk_text = f"[{chunk.source}]\n{chunk.text}\n\n"
            chunk_tokens = self.estimate_tokens(chunk_text)

            # 如果加上当前chunk会超出限制，需要截断
            if total_tokens + chunk_tokens > max_available_tokens:
                # 计算剩余可使用的token数
                remaining_tokens = max_available_tokens - total_tokens
                if remaining_tokens <= 0:
                    break

                # 在chunk的后10%文本中找句号，确保在完整句子处截断
                encoded = self.tokenizer.encode(chunk_text, add_special_tokens=False)
                if len(encoded) > remaining_tokens:
                    # 先按token数截断，再在截断文本中找最后一个句号
                    truncated_encoded = encoded[:remaining_tokens]
                    truncated_text = self.tokenizer.decode(truncated_encoded, skip_special_tokens=True)
                    # 找最后一个句号，避免句子被截断
                    last_period = truncated_text.rfind(".")
                    if last_period != -1:
                        truncated_text = truncated_text[:last_period + 1]
                    context_parts.append(truncated_text + "\n\n")
                break

            # 未超出限制，直接添加
            context_parts.append(chunk_text)
            total_tokens += chunk_tokens

        return "".join(context_parts).strip()

    def _analyze_sources(self, chunks: List[DocumentChunk]) -> Dict[str, int]:
        """统计来源分布，补充元数据"""
        source_counts = {}
        for chunk in chunks:
            source = chunk.source
            source_counts[source] = source_counts.get(source, 0) + 1
        return source_counts

    def assemble_context(self, retrieved_docs: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        上下文组装主方法：整合去重、排序、截断，返回最终上下文和元数据
        :param retrieved_docs: 重排序后的检索结果（和你之前reranker的输出格式兼容）
        :return: 包含最终上下文、元数据、选中chunk的字典
        """
        # 1. 转换文档格式为DocumentChunk
        chunks = []
        for doc in retrieved_docs:
            chunk = DocumentChunk(
                text=doc["content"],
                metadata=doc.get("meta_data", {}),
                relevance_score=doc["final_score"],  # 直接用reranker输出的综合分数
                source=doc["meta_data"].get("source", "unknown"),
                chunk_id=doc.get("chunk_id", f"chunk_{len(chunks)}")
            )
            chunks.append(chunk)

        # 2. 文本去重
        unique_chunks = self.deduplicate_chunks(chunks)

        # 3. 优先级排序，保证多样性
        prioritized_chunks = self.prioritize_chunks(unique_chunks)

        # 4. 构建并截断上下文
        final_context = self.truncate_context(prioritized_chunks)

        # 5. 补充元数据
        context_metadata = {
            "total_chunks_retrieved": len(retrieved_docs),
            "unique_chunks_after_dedup": len(unique_chunks),
            "chunks_selected": len(prioritized_chunks),
            "estimated_tokens": self.estimate_tokens(final_context),
            "chunk_sources": self._analyze_sources(prioritized_chunks)
        }

        return {
            "context_text": final_context,
            "metadata": context_metadata,
            "selected_chunks": prioritized_chunks
        }

# 使用示例（和你之前的reranker输出兼容）
if __name__ == "__main__":
    # 模拟你之前reranker的输出结果
    reranked_results = [
        {
            "content": "MI patients have higher incidence of heart failure and coronary artery disease.",
            "meta_data": {"source": "Circulation 2024", "topic": "MI Complications"},
            "final_score": 0.8498,
            "chunk_id": "chunk_1"
        },
        {
            "content": "Myocardial infarction (MI) is a major risk factor for cardiovascular disease.",
            "meta_data": {"source": "Cardiology Journal 2024", "topic": "MI & CVD"},
            "final_score": 0.7725,
            "chunk_id": "chunk_2"
        }
    ]

    # 初始化上下文组装器（用本地tokenizer，适配无网环境）
    assembler = ContextAssembler(
        tokenizer_name_or_path="/root/.cache/huggingface/hub/models--BAAI--bge-small-en-v1.5/snapshots/latest/",
        max_context_tokens=2048
    )

    # 组装上下文
    result = assembler.assemble_context(reranked_results)

    # 打印结果
    print("="*50)
    print("最终上下文：")
    print(result["context_text"])
    print("="*50)
    print("元数据：")
    print(result["metadata"])

最终上下文：
[Circulation 2024]
MI patients have higher incidence of heart failure and coronary artery disease.

[Cardiology Journal 2024]
Myocardial infarction (MI) is a major risk factor for cardiovascular disease.
元数据：
{'total_chunks_retrieved': 2, 'unique_chunks_after_dedup': 2, 'chunks_selected': 2, 'estimated_tokens': 44, 'chunk_sources': {'Circulation 2024': 1, 'Cardiology Journal 2024': 1}}


In [5]:
from dataclasses import dataclass
from typing import Dict, Any, Optional

# 1. 定义提示词阶段数据类
@dataclass
class PromptStage:
    name: str
    system_prompt: str
    user_prompt_template: str
    temperature: float
    max_tokens: int

# 2. 医学提示工程模板
class MedicalPromptEngineer:
    def __init__(self):
        # 初始化各阶段提示词模板（适配医学场景，可直接使用）
        self.stages: Dict[str, PromptStage] = {
            "evidence_evaluator": PromptStage(
                name="证据评估器",
                system_prompt=(
                    "你是一名严谨的医学证据评估专家。你的任务是评估用户提供的医学文献证据与问题的相关性、可靠性和局限性。\n"
                    "请基于提供的上下文，对证据进行分级，并指出证据的优点、不足和适用范围。"
                ),
                user_prompt_template=(
                    "用户问题：{query}\n"
                    "参考证据：{context}\n"
                    "请完成以下评估：\n"
                    "1. 证据与问题的相关性（高/中/低）及理由\n"
                    "2. 证据的可靠性（高/中/低）及理由\n"
                    "3. 证据的局限性和适用人群\n"
                    "4. 是否存在矛盾或冲突的信息"
                ),
                temperature=0.2,
                max_tokens=512
            ),
            "answer_generator": PromptStage(
                name="答案生成器",
                system_prompt=(
                    "你是一名专业的医学顾问，需基于用户问题和评估后的医学证据，生成准确、专业、易于理解的回答。\n"
                    "回答需严格依据提供的证据，避免编造信息；对于不确定的内容，需明确标注。"
                ),
                user_prompt_template=(
                    "用户问题：{query}\n"
                    "评估后的证据：{evaluated_evidence}\n"
                    "请生成一份清晰、结构化的回答，包括：\n"
                    "1. 问题核心结论\n"
                    "2. 关键证据支持\n"
                    "3. 注意事项和建议"
                ),
                temperature=0.3,
                max_tokens=1024
            ),
            "critical_reviewer": PromptStage(
                name="批判性审查器",
                system_prompt=(
                    "你是一名医学批判性审查专家。你的任务是对生成的回答进行审查，检查是否存在错误、偏见、夸大或信息缺失。\n"
                    "请从医学严谨性、证据一致性、表述准确性三个维度进行审查，并提出修改建议。"
                ),
                user_prompt_template=(
                    "用户问题：{query}\n"
                    "生成的回答：{draft_answer}\n"
                    "请进行批判性审查：\n"
                    "1. 回答是否与证据一致？\n"
                    "2. 是否存在医学错误或误导性表述？\n"
                    "3. 是否遗漏了关键信息？\n"
                    "4. 提出具体修改建议"
                ),
                temperature=0.1,
                max_tokens=512
            ),
            "final_assembler": PromptStage(
                name="最终组装器",
                system_prompt=(
                    "你是一名医学回答优化专家。你的任务是根据批判性审查意见，优化回答，使其更准确、严谨、完整，同时保持语言通俗易懂。"
                ),
                user_prompt_template=(
                    "用户问题：{query}\n"
                    "原始回答：{draft_answer}\n"
                    "审查意见：{review_comments}\n"
                    "请生成最终版本的回答，确保：\n"
                    "1. 修正所有指出的问题\n"
                    "2. 保持医学准确性\n"
                    "3. 语言清晰、结构合理\n"
                    "4. 标注信息来源"
                ),
                temperature=0.2,
                max_tokens=1024
            )
        }

    def get_prompt(self, stage_name: str, **kwargs) -> Dict[str, Any]:
        """
        获取指定阶段的完整提示词
        :param stage_name: 阶段名称，如 'evidence_evaluator'
        :param kwargs: 模板填充参数，如 query、context、evaluated_evidence 等
        :return: 包含 system_prompt、user_prompt、temperature、max_tokens 的字典
        """
        if stage_name not in self.stages:
            raise ValueError(f"不存在的提示词阶段：{stage_name}")

        stage = self.stages[stage_name]
        user_prompt = stage.user_prompt_template.format(**kwargs)

        return {
            "system_prompt": stage.system_prompt,
            "user_prompt": user_prompt,
            "temperature": stage.temperature,
            "max_tokens": stage.max_tokens,
            "stage_name": stage.name
        }

# 使用示例（和上下文组装器的输出无缝衔接）
if __name__ == "__main__":
    # 1. 模拟上下文组装器的输出（你上一步的结果）
    assembled_context = {
        "context_text": (
            "[Circulation 2024]\n"
            "MI patients have higher incidence of heart failure and coronary artery disease.\n\n"
            "[Cardiology Journal 2024]\n"
            "Myocardial infarction (MI) is a major risk factor for cardiovascular disease."
        )
    }
    query = "MI对心血管疾病有何影响？"

    # 2. 初始化提示工程模块
    prompt_engineer = MedicalPromptEngineer()

    # 3. 生成「证据评估器」的提示词（直接喂给大模型即可）
    evidence_prompt = prompt_engineer.get_prompt(
        "evidence_evaluator",
        query=query,
        context=assembled_context["context_text"]
    )

    # 打印提示词
    print("="*60)
    print(f"【阶段】{evidence_prompt['stage_name']}")
    print("="*60)
    print("System Prompt:")
    print(evidence_prompt["system_prompt"])
    print("\nUser Prompt:")
    print(evidence_prompt["user_prompt"])
    print("\n参数配置：")
    print(f"Temperature: {evidence_prompt['temperature']}")
    print(f"Max Tokens: {evidence_prompt['max_tokens']}")

【阶段】证据评估器
System Prompt:
你是一名严谨的医学证据评估专家。你的任务是评估用户提供的医学文献证据与问题的相关性、可靠性和局限性。
请基于提供的上下文，对证据进行分级，并指出证据的优点、不足和适用范围。

User Prompt:
用户问题：MI对心血管疾病有何影响？
参考证据：[Circulation 2024]
MI patients have higher incidence of heart failure and coronary artery disease.

[Cardiology Journal 2024]
Myocardial infarction (MI) is a major risk factor for cardiovascular disease.
请完成以下评估：
1. 证据与问题的相关性（高/中/低）及理由
2. 证据的可靠性（高/中/低）及理由
3. 证据的局限性和适用人群
4. 是否存在矛盾或冲突的信息

参数配置：
Temperature: 0.2
Max Tokens: 512


In [6]:
#测试
#from transformers import AutoTokenizer, AutoModelForCausalLM
#model_path = "/root/.cache/huggingface/hub/models--deepseek-ai--deepseek-llm-7b-chat/snapshots/afbda8b347ec881666061fa67447046fc5164ec8"
#tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
#model = AutoModelForCausalLM.from_pretrained(
    #model_path,
    #device_map="auto",
    #trust_remote_code=True)
#prompt = "Hello, how are you?"
#inputs = tokenizer(prompt, return_tensors="pt")
#inputs = inputs.to(model.device)
#outputs = model.generate(**inputs, max_new_tokens=50)
# 解码时强制处理空格符号
#response = tokenizer.decode(
    #outputs[0],
    #skip_special_tokens=True,
    #clean_up_tokenization_spaces=True  # 关键：自动处理 Ġ 空格)
# 额外做一次清理，把残留的符号也去掉
#response = response.replace("Ġ", " ").strip()
#print(response)

In [7]:
#本地LLM集成与生成
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/root/.cache/huggingface/hub/models--deepseek-ai--deepseek-llm-7b-chat/snapshots/afbda8b347ec881666061fa67447046fc5164ec8"
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

messages = [
    {"role": "system", "content": "你是一个专业的医疗助手"},
    {"role": "user", "content": "mi对心血管疾病有何影响？"}
]

# 1. 生成输入，保持字典格式
inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt"
).to(model.device)

# 2. 把字典拆开，单独提取 input_ids 用于生成
input_ids = inputs["input_ids"]
attention_mask = inputs.get("attention_mask")

# 3. 模型生成
outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

# 4. 解码：只取模型新生成的部分，同时处理Ġ乱码
response = tokenizer.decode(
    outputs[0][input_ids.shape[1]:],  # 用 input_ids 的长度来截取
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)

# 只保留英文、数字、空格和常用标点
response = re.sub(r'[^a-zA-Z0-9\s.,!?\'"-]', '', response)
# 再清一下多余空格
response = ' '.join(response.split())

print("\n【模型回答】\n", response.strip())

Loading weights: 100%|██████████| 273/273 [00:08<00:00, 30.84it/s]



【模型回答】
 robothello!howcanihelpyoutoday?welcometotheMirobot!PleaseletmeknowhowIcanhelpyou.Youcanusethecommands!boardtocreateanewboard,!posttoaddanewpost,and!deletetodeleteapost.Youcanalsousethecommands!edittoeditapost,!movetomoveapost,and!renametorenameaboard.Ifyouhaveanyquestionsorneedhelpwithanyofthesecommands,feelfreetoask!LetmeknowifthereisanythingelseIcanhelpyouwith.Haveagreatday!


In [13]:
import time
from typing import List, Dict, Any

# 1. LLM 生成器
class FakerLLMGenerator:
    """模拟 LLM 生成，返回固定的医疗回答，格式和真实模型一致"""
    def __init__(self):
        pass

    def generate(self, prompt: str) -> str:
        """
        这里你可以把 prompt 打印出来，确认上下文和模板拼接是否正确，
        也可以根据不同的 query 返回不同的假答案。
        """
        print(f"\n[LLM Prompt Preview]\n{prompt}\n")
        return (
            "根据提供的医学资料，mi类药物对心血管疾病的影响主要包括：\n"
            "1. 降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗。\n"
            "2. 具有轻度血管扩张作用，可改善外周循环，缓解部分患者的症状。\n"
            "3. 长期使用可能影响电解质平衡，建议定期监测血钾水平。\n"
            "4. 部分患者可能出现心动过缓、乏力等不良反应，需在医生指导下调整剂量。"
        )

# 2. 上下文组装器
class ContextAssembler:
    def assemble(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """
        模拟你的上下文组装器，返回格式：
        {
            "selected_chunks": [...],
            "context_str": "...",
            "metadata": {...}
        }
        """
        context_str = "\n\n".join([
            f"【文档 {i+1}】{chunk['title']}\n{chunk['content']}"
            for i, chunk in enumerate(retrieved_chunks)
        ])
        return {
            "selected_chunks": retrieved_chunks,
            "context_str": context_str,
            "metadata": {
                "total_chunks": len(retrieved_chunks),
                "relevance_scores": [0.95, 0.88, 0.82]
            }
        }

# 3. 医学提示词模板（你已经完成的部分）
class MedicalPromptTemplate:
    def build(self, query: str, context: str) -> str:
        """按你的医学模板拼接 prompt"""
        system_prompt = (
            "你是一名专业的医疗助手，请根据提供的医学资料回答用户问题。\n"
            "回答要求：\n"
            "1. 基于上下文信息，内容准确、严谨\n"
            "2. 语言通俗易懂，结构清晰\n"
            "3. 若信息不足，请明确说明，不要编造内容\n"
        )
        user_prompt = f"用户问题：{query}\n参考资料：\n{context}"
        return f"{system_prompt}\n{user_prompt}"

# 4. 核心流程类：MedicalGenerationPipeline
class MedicalGenerationPipeline:
    def __init__(
        self,
        context_assembler: ContextAssembler,
        prompt_template: MedicalPromptTemplate,
        llm_generator: FakerLLMGenerator
    ):
        # 初始化组件
        self.context_assembler = context_assembler
        self.prompt_template = prompt_template
        self.llm_generator = llm_generator

    def _format_sources(self, chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """格式化引用来源"""
        return [
            {
                "doc_id": chunk["doc_id"],
                "title": chunk["title"],
                "pages": chunk.get("pages", "未标注页码")
            }
            for chunk in chunks
        ]

    def _add_disclaimer(self, answer: str) -> str:
        """添加免责声明和格式美化"""
        disclaimer = "\n\n【免责声明】本回答仅供参考，不构成医疗建议，请咨询专业医师。"
        return answer + disclaimer

    def run(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        start_time = time.time()
        stage_times = {}
        stage_success = []

        # --------------------------
        # 上下文组装
        # --------------------------
        t0 = time.time()
        context_result = self.context_assembler.assemble(query, retrieved_chunks)
        stage_times["context_assembly"] = time.time() - t0
        stage_success.append(True)

        # --------------------------
        # 生成答案草稿
        # --------------------------
        t1 = time.time()
        prompt = self.prompt_template.build(query, context_result["context_str"])
        draft_answer = self.llm_generator.generate(prompt)
        stage_times["draft_generation"] = time.time() - t1
        stage_success.append(True)

        # --------------------------
        # 生成最终答案（跳过可选的步骤4）
        # --------------------------
        final_answer = draft_answer

        # --------------------------
        # 后处理（引用标记、格式美化、免责声明）
        # --------------------------
        t2 = time.time()
        final_answer = self._add_disclaimer(final_answer)
        stage_times["post_processing"] = time.time() - t2
        stage_success.append(True)

        # 组装最终结果
        total_time = time.time() - start_time
        result = {
            "query": query,
            "answer": final_answer,
            "context_metadata": context_result["metadata"],
            "generation_metrics": {
                "total_time_seconds": round(total_time, 2),
                "stage_times": stage_times,
                "token_counts": {
                    "prompt_tokens": len(prompt.split()),
                    "completion_tokens": len(final_answer.split())
                },
                "stage_success": stage_success
            },
            "intermediate_results": {
                "evidence_evaluation": "（可选）上下文相关性良好，可支撑回答",
                "draft_answer": draft_answer,
                "review_feedback": "（可选）无审查，直接使用草稿"
            },
            "sources": self._format_sources(context_result["selected_chunks"]),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        return result

# 5. 测试完整流程
if __name__ == "__main__":
    # 模拟检索到的文档块（你的 RAG 组件输出的格式）
    test_chunks = [
        {
            "doc_id": "doc_001",
            "title": "心血管疾病药物治疗指南",
            "content": "mi类药物可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，长期使用需监测血钾。",
            "pages": "12-15"
        },
        {
            "doc_id": "doc_003",
            "title": "临床药理学手册",
            "content": "mi类药物具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状。",
            "pages": "89-92"
        }
    ]

    # 初始化组件
    assembler = ContextAssembler()
    template = MedicalPromptTemplate()
    fake_llm = FakerLLMGenerator()

    # 初始化 pipeline
    pipeline = MedicalGenerationPipeline(assembler, template, fake_llm)

    # 测试 query
    test_query = "mi对心血管疾病有何影响？"

    # 运行流程
    print("=== 测试开始 ===")
    result = pipeline.run(test_query, test_chunks)

    # 打印结果
    print("\n=== 最终回答 ===")
    print(result["answer"])

    print("\n=== 来源信息 ===")
    for source in result["sources"]:
        print(f"- 来源：{source['title']}（{source['pages']}）")

    print("\n=== 关键指标 ===")
    print(f"总耗时：{result['generation_metrics']['total_time_seconds']}s")
    print(f"阶段耗时：{result['generation_metrics']['stage_times']}")
    print(f"Token 数：{result['generation_metrics']['token_counts']}")
    print(f"阶段状态：{result['generation_metrics']['stage_success']}")

    print("\n=== 完整 JSON 结果 ===")
    import json
    print(json.dumps(result, ensure_ascii=False, indent=2))

=== 测试开始 ===

[LLM Prompt Preview]
你是一名专业的医疗助手，请根据提供的医学资料回答用户问题。
回答要求：
1. 基于上下文信息，内容准确、严谨
2. 语言通俗易懂，结构清晰
3. 若信息不足，请明确说明，不要编造内容

用户问题：mi对心血管疾病有何影响？
参考资料：
【文档 1】心血管疾病药物治疗指南
mi类药物可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，长期使用需监测血钾。

【文档 2】临床药理学手册
mi类药物具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状。


=== 最终回答 ===
根据提供的医学资料，mi类药物对心血管疾病的影响主要包括：
1. 降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗。
2. 具有轻度血管扩张作用，可改善外周循环，缓解部分患者的症状。
3. 长期使用可能影响电解质平衡，建议定期监测血钾水平。
4. 部分患者可能出现心动过缓、乏力等不良反应，需在医生指导下调整剂量。

【免责声明】本回答仅供参考，不构成医疗建议，请咨询专业医师。

=== 来源信息 ===
- 来源：心血管疾病药物治疗指南（12-15）
- 来源：临床药理学手册（89-92）

=== 关键指标 ===
总耗时：0.0s
阶段耗时：{'context_assembly': 2.9802322387695312e-05, 'draft_generation': 3.4809112548828125e-05, 'post_processing': 3.814697265625e-06}
Token 数：{'prompt_tokens': 16, 'completion_tokens': 10}
阶段状态：[True, True, True]

=== 完整 JSON 结果 ===
{
  "query": "mi对心血管疾病有何影响？",
  "answer": "根据提供的医学资料，mi类药物对心血管疾病的影响主要包括：\n1. 降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗。\n2. 具有轻度血管扩张作用，可改善外周循环，缓解部分患者的症状。\n3. 长期使用可能影响电解质平衡，建议定期监测血钾水平。\n4. 部分患者可能出现心动过缓、乏力等不良反应，需在医生指导下调整剂量。\n\n【免责声明】本回答仅供参考，不

In [11]:
# 答案评估模块（可直接运行，基于现有的答案）
import json
import re
from rouge_score import rouge_scorer

class AnswerEvaluator:
    def __init__(self):
        # 医疗关键词模板
        self.medical_keywords = {"mi", "心肌", "心绞痛", "血管扩张", "心率", "血钾", "心血管"}
        
        # ROUGE 评估器
        self.rouge = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

        # 标准参考答案
        self.reference_answer = (
            "mi类药物可降低心率与心肌耗氧量，用于心绞痛治疗，具有血管扩张作用，"
            "长期使用需监测血钾，可能出现心动过缓、乏力等不良反应。"
        )

    def calculate_rouge_score(self, answer):
        """计算 ROUGE 相似度分数"""
        scores = self.rouge.score(self.reference_answer, answer)
        return {
            "rouge1_precision": round(scores["rouge1"].precision, 2),
            "rouge1_recall": round(scores["rouge1"].recall, 2),
            "rougeL_f1": round(scores["rougeL"].fmeasure, 2)
        }

    def check_key_medical_info(self, answer):
        """关键医学信息覆盖率"""
        found = [k for k in self.medical_keywords if k in answer]
        coverage = round(len(found) / len(self.medical_keywords), 2)
        return {
            "found_keywords": found,
            "coverage_rate": coverage
        }

    def check_hallucination(self, answer):
        """简单幻觉检测（无参考则判定安全）"""
        forbidden = ["研究表明", "专家认为", "据统计", "临床试验"]
        has_hallucination = any(p in answer for p in forbidden)
        return {
            "has_hallucination": has_hallucination,
            "hallucination_risk": "低" if not has_hallucination else "中"
        }

    def evaluate(self, answer):
        """执行完整评估"""
        rouge_scores = self.calculate_rouge_score(answer)
        key_info = self.check_key_medical_info(answer)
        hallucination = self.check_hallucination(answer)

        # 综合评分（0~100）
        total_score = int(
            rouge_scores["rougeL_f1"] * 40
            + key_info["coverage_rate"] * 50
            + (0 if hallucination["has_hallucination"] else 10)
        )

        return {
            "total_score": total_score,
            "rouge_scores": rouge_scores,
            "medical_key_info": key_info,
            "hallucination_check": hallucination
        }

# 测试：直接评估你现有的答案
if __name__ == "__main__":
    # 你当前的假答案（直接用你 pipeline 输出的 answer）
    test_answer = """根据提供的医学资料，mi类药物对心血管疾病的影响主要包括：
1. 降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗。
2. 具有轻度血管扩张作用，可改善外周循环，缓解部分患者的症状。
3. 长期使用可能影响电解质平衡，建议定期监测血钾水平。
4. 部分患者可能出现心动过缓、乏力等不良反应，需在医生指导下调整剂量。

【免责声明】本回答仅供参考，不构成医疗建议，请咨询专业医师。"""

    evaluator = AnswerEvaluator()
    eval_result = evaluator.evaluate(test_answer)

    print("===== 答案评估结果 =====")
    print(json.dumps(eval_result, ensure_ascii=False, indent=2))

===== 答案评估结果 =====
{
  "total_score": 73,
  "rouge_scores": {
    "rouge1_precision": 0.2,
    "rouge1_recall": 1.0,
    "rougeL_f1": 0.33
  },
  "medical_key_info": {
    "found_keywords": [
      "mi",
      "心肌",
      "心率",
      "心绞痛",
      "血钾",
      "心血管",
      "血管扩张"
    ],
    "coverage_rate": 1.0
  },
  "hallucination_check": {
    "has_hallucination": false,
    "hallucination_risk": "低"
  }
}


In [38]:
import json
import time
import hashlib
from typing import List, Dict, Any, Tuple
from concurrent.futures import ThreadPoolExecutor

# 1.  LLM 生成器
class FakerLLMGenerator:
    def generate(self, prompt: str, temperature: float = 0.1) -> str:
        print(f"\n[LLM Prompt Preview]\n{prompt}\n")
        return (
            "根据提供的医学资料，mi类药物对心血管疾病的影响主要包括：\n"
            "1. 降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗。\n"
            "2. 具有轻度血管扩张作用，可改善外周循环，缓解部分患者的症状。\n"
            "3. 长期使用可能影响电解质平衡，建议定期监测血钾水平。"
        )

# 2. 上下文组装器
class ContextAssembler:
    def assemble(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        context_str = "\n\n".join([
            f"【文档 {i+1}】{chunk['title']}\n{chunk['content']}"
            for i, chunk in enumerate(retrieved_chunks)
        ])
        return {
            "selected_chunks": retrieved_chunks,
            "context_str": context_str,
            "metadata": {"total_chunks": len(retrieved_chunks)}
        }

# 3. 医学提示词模板
class MedicalPromptTemplate:
    def build(self, query: str, context: str) -> str:
        system_prompt = (
            "你是一名专业的医疗助手，请根据提供的医学资料回答用户问题。\n"
            "回答要求：内容准确、严谨，通俗易懂，结构清晰。"
        )
        return f"{system_prompt}\n用户问题：{query}\n参考资料：\n{context}"


# 4. 缓存管理器（查询+上下文哈希、TTL、温度限制）
class LLMResultCache:
    def __init__(self, ttl_seconds: int = 3600, max_size: int = 1000):
        self.cache: Dict[str, Dict[str, Any]] = {}
        self.ttl = ttl_seconds
        self.max_size = max_size

    def _get_cache_key(self, query: str, context: str) -> str:
        combined_str = query + "|" + context
        return hashlib.md5(combined_str.encode()).hexdigest()

    def get(self, query: str, context: str) -> str | None:
        key = self._get_cache_key(query, context)
        if key not in self.cache:
            return None
        entry = self.cache[key]
        if time.time() - entry["timestamp"] > self.ttl:
            del self.cache[key]
            return None
        return entry["result"]

    def set(self, query: str, context: str, result: str, temperature: float):
        # 只缓存低温（确定性）结果
        if temperature > 0.2:
            return
        # 大小限制，避免内存溢出
        if len(self.cache) >= self.max_size:
            oldest_key = min(self.cache, key=lambda k: self.cache[k]["timestamp"])
            del self.cache[oldest_key]
        key = self._get_cache_key(query, context)
        self.cache[key] = {"result": result, "timestamp": time.time()}

# 5. 核心生成器
class MedicalGenerationService:
    def __init__(
        self,
        context_assembler: ContextAssembler,
        prompt_template: MedicalPromptTemplate,
        llm_generator: FakeLLMGenerator,
        cache: LLMResultCache
    ):
        self.context_assembler = context_assembler
        self.prompt_template = prompt_template
        self.llm_generator = llm_generator
        self.cache = cache

    def run(self, query: str, retrieved_chunks: List[Dict[str, Any]], temperature: float = 0.1) -> Dict[str, Any]:
        start_time = time.time()
        prompt = ""

        # 1. 上下文组装
        context_result = self.context_assembler.assemble(query, retrieved_chunks)

        # 2. 缓存命中检查
        cached_answer = self.cache.get(query, context_result["context_str"])
        if cached_answer:
            draft_answer = cached_answer
            print("[Cache Hit] 使用缓存结果")
            prompt = self.prompt_template.build(query, context_result["context_str"])
        else:
            # 3. 生成草稿
            prompt = self.prompt_template.build(query, context_result["context_str"])
            draft_answer = self.llm_generator.generate(prompt, temperature)
            self.cache.set(query, context_result["context_str"], draft_answer, temperature)

        # 4. 后处理（格式美化+免责声明）
        final_answer = draft_answer + "\n\n【免责声明】本回答仅供参考，不构成医疗建议，请咨询专业医师。"

        # 组装结果
        return {
            "query": query,
            "answer": final_answer,
            "context_metadata": context_result["metadata"],
            "sources": [{"doc_id": c["doc_id"], "title": c["title"]} for c in context_result["selected_chunks"]],
            "metrics": {
                "total_time": round(time.time() - start_time, 2),
                "prompt_tokens": len(prompt.split()),
                "completion_tokens": len(final_answer.split())
            },
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }

    # 批量处理
    def batch_run(self, queries_with_chunks: List[Tuple[str, List[Dict[str, Any]]]], max_workers: int = 4) -> List[Dict[str, Any]]:
        results = [None] * len(queries_with_chunks)
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for idx, (query, chunks) in enumerate(queries_with_chunks):
                future = executor.submit(self.run, query, chunks)
                futures.append((idx, future))

            for idx, future in futures:
                try:
                    results[idx] = future.result()
                except Exception as e:
                    print(f"任务 {idx} 失败: {e}")
                    results[idx] = {"query": queries_with_chunks[idx][0], "error": str(e)}
        return results

# 测试运行
if __name__ == "__main__":
    # 测试数据
    test_chunks = [
        {
            "doc_id": "doc_001",
            "title": "心血管疾病药物治疗指南",
            "content": "mi类药物可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，长期使用需监测血钾。"
        },
        {
            "doc_id": "doc_003",
            "title": "临床药理学手册",
            "content": "mi类药物具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状。"
        }
    ]
    test_queries = [
        ("mi对心血管疾病有何影响？", test_chunks),
        ("mi类药物的主要作用是什么？", test_chunks)
    ]

    # 初始化组件
    assembler = ContextAssembler()
    template = MedicalPromptTemplate()
    llm = FakeLLMGenerator()
    cache = LLMResultCache(ttl_seconds=3600)
    service = MedicalGenerationService(assembler, template, llm, cache)

    print("=== 单次测试（含缓存验证） ===")
    result1 = service.run(test_queries[0][0], test_queries[0][1])
    print(f"第一次运行耗时: {result1['metrics']['total_time']}s")

    print("\n=== 再次运行同一 query（应命中缓存） ===")
    result2 = service.run(test_queries[0][0], test_queries[0][1])
    print(f"第二次运行耗时: {result2['metrics']['total_time']}s")

    print("\n=== 批量处理测试 ===")
    batch_results = service.batch_run(test_queries)
    for i, res in enumerate(batch_results):
        print(f"\n【任务 {i}】Query: {res['query']}")
        if "error" in res:
            print(f"错误: {res['error']}")
        else:
            print(f"耗时: {res['metrics']['total_time']}s")
            print(f"答案片段: {res['answer'][:100]}...")

=== 单次测试（含缓存验证） ===

[LLM Prompt Preview]
你是一名专业的医疗助手，请根据提供的医学资料回答用户问题。
回答要求：内容准确、严谨，通俗易懂，结构清晰。
用户问题：mi对心血管疾病有何影响？
参考资料：
【文档 1】心血管疾病药物治疗指南
mi类药物可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，长期使用需监测血钾。

【文档 2】临床药理学手册
mi类药物具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状。

第一次运行耗时: 0.0s

=== 再次运行同一 query（应命中缓存） ===
[Cache Hit] 使用缓存结果
第二次运行耗时: 0.0s

=== 批量处理测试 ===
[Cache Hit] 使用缓存结果

[LLM Prompt Preview]
你是一名专业的医疗助手，请根据提供的医学资料回答用户问题。
回答要求：内容准确、严谨，通俗易懂，结构清晰。
用户问题：mi类药物的主要作用是什么？
参考资料：
【文档 1】心血管疾病药物治疗指南
mi类药物可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，长期使用需监测血钾。

【文档 2】临床药理学手册
mi类药物具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状。


【任务 0】Query: mi对心血管疾病有何影响？
耗时: 0.0s
答案片段: 根据提供的医学资料，mi类药物对心血管疾病的影响主要包括：
1. 降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗。
2. 具有轻度血管扩张作用，可改善外周循环，缓解部分患者的症状。
3. 长期使用可能...

【任务 1】Query: mi类药物的主要作用是什么？
耗时: 0.0s
答案片段: 根据提供的医学资料，mi类药物对心血管疾病的影响主要包括：
1. 降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗。
2. 具有轻度血管扩张作用，可改善外周循环，缓解部分患者的症状。
3. 长期使用可能...


In [14]:
import re
from typing import List, Dict, Any

# 强约束系统提示词层实现
class StrongConstraintSystem:
    def __init__(self):
        # 医学术语缩写-全称映射（可按需扩展）
        self.medical_abbr_map = {
            "mi": "β受体阻滞剂（mi类药物）",
            "ACEI": "血管紧张素转换酶抑制剂",
            "ARB": "血管紧张素Ⅱ受体拮抗剂"
        }
        # 引用正则：匹配 [文献1]、[文献2] 等格式
        self.citation_pattern = re.compile(r"\[文献(\d+)\]")

    # a. 知识库边界提示词
    def build_boundary_prompt(self) -> str:
        return (
            "【知识库边界规则】\n"
            "你只能根据提供的参考资料回答问题，严禁编造内容。\n"
            "如果问题无法从参考资料中找到答案，请明确回答：「根据现有文献无法回答此问题」。\n"
        )

    # b. 引用来源提示词 + 校验器
    def build_citation_prompt(self, doc_ids: List[int]) -> str:
        doc_list = "\n".join([f"[文献{i}] 为本次参考资料的第{i}篇文档" for i in doc_ids])
        return (
            "【引用来源规则】\n"
            "1. 回答中涉及的每一个事实、数据、结论，都必须在句末标注对应的来源，格式为：[文献N]\n"
            "2. 只能使用本次提供的文献编号作为引用，不得编造不存在的文献编号\n"
            "3. 参考资料列表：\n"
            f"{doc_list}\n"
        )

    def validate_citations(self, answer: str, valid_doc_ids: List[int]) -> Dict[str, Any]:
        """检查引用是否有效"""
        found_citations = set(self.citation_pattern.findall(answer))
        valid_ids = set(str(i) for i in valid_doc_ids)
        invalid_citations = found_citations - valid_ids
        missing_citations = valid_ids - found_citations

        return {
            "has_invalid_citation": len(invalid_citations) > 0,
            "invalid_citations": list(invalid_citations),
            "missing_citations": list(missing_citations),
            "is_citation_valid": len(invalid_citations) == 0
        }

    # c. 禁止编造提示词
    def build_no_fabrication_prompt(self) -> str:
        return (
            "【禁止编造规则】\n"
            "1. 严禁添加参考资料中未提及的数据、结论、百分比或细节\n"
            "2. 严禁使用「研究表明」「已被证明」「100%有效」等无依据表述\n"
            "3. 回答中的所有信息必须能在参考资料中找到原文对应内容\n"
        )

    # d. 术语规范与输出格式提示词 + 格式校验器
    def build_format_prompt(self) -> str:
        return (
            "【输出格式与术语规范】\n"
            "1. 医学术语缩写首次出现时，必须给出全称，例如：β受体阻滞剂（mi类药物）\n"
            "2. 回答必须包含以下固定章节标题：\n"
            "   - 核心答案\n"
            "   - 证据总结\n"
            "   - 参考文献\n"
            "3. 参考文献部分需列出本次引用的所有文献，格式为：[文献N] 文档标题（年份，来源）\n"
        )

    def validate_format(self, answer: str) -> Dict[str, Any]:
        """检查输出格式是否符合要求"""
        required_sections = ["核心答案", "证据总结", "参考文献"]
        found_sections = [s for s in required_sections if s in answer]
        missing_sections = list(set(required_sections) - set(found_sections))

        # 检查缩写是否已扩展（简单规则：缩写出现时是否伴随全称）
        abbr_errors = []
        for abbr, full in self.medical_abbr_map.items():
            if abbr in answer and full not in answer:
                abbr_errors.append(abbr)

        return {
            "missing_sections": missing_sections,
            "abbr_errors": abbr_errors,
            "is_format_valid": len(missing_sections) == 0 and len(abbr_errors) == 0
        }

    # 构建完整强约束提示词
    def build_full_constraint_prompt(self, doc_ids: List[int]) -> str:
        parts = [
            self.build_boundary_prompt(),
            self.build_citation_prompt(doc_ids),
            self.build_no_fabrication_prompt(),
            self.build_format_prompt()
        ]
        return "\n".join(parts)

    # 回答校验总入口
    def validate_answer(self, answer: str, valid_doc_ids: List[int]) -> Dict[str, Any]:
        citation_check = self.validate_citations(answer, valid_doc_ids)
        format_check = self.validate_format(answer)
        return {
            "citation_check": citation_check,
            "format_check": format_check,
            "is_all_valid": citation_check["is_citation_valid"] and format_check["is_format_valid"]
        }

# 测试：与现有流程集成
if __name__ == "__main__":
    # 1. 初始化强约束系统
    constraint_system = StrongConstraintSystem()

    # 2. 测试用例：上下文文档与生成的回答
    test_chunks = [
        {"doc_id": 1, "title": "心血管疾病药物治疗指南", "content": "mi类药物可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，长期使用需监测血钾。", "year": 2023, "source": "临床药学杂志"},
        {"doc_id": 2, "title": "临床药理学手册", "content": "mi类药物具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状。", "year": 2022, "source": "人民卫生出版社"}
    ]
    valid_doc_ids = [1, 2]

    # 3. 构建带强约束的提示词
    base_prompt = "用户问题：mi对心血管疾病有何影响？\n参考资料：\n" + "\n".join([
        f"[文献{i+1}] {chunk['title']}\n{chunk['content']}"
        for i, chunk in enumerate(test_chunks)
    ])
    constraint_prompt = constraint_system.build_full_constraint_prompt(valid_doc_ids)
    full_prompt = f"{constraint_prompt}\n\n{base_prompt}"

    print("=== 带强约束的完整提示词 ===")
    print(full_prompt)

    # 4. 模拟生成回答并校验
    test_answer = """
核心答案
β受体阻滞剂（mi类药物）可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，同时具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状[文献1][文献2]。

证据总结
1. 心率与心肌耗氧量降低：基于《心血管疾病药物治疗指南》的推荐，该类药物是稳定型心绞痛的一线治疗药物[文献1]。
2. 外周循环改善：《临床药理学手册》指出，其血管扩张作用可缓解部分患者症状[文献2]。
3. 用药安全：长期使用需监测血钾水平，避免电解质紊乱[文献1]。

参考文献
[文献1] 心血管疾病药物治疗指南（2023，临床药学杂志）
[文献2] 临床药理学手册（2022，人民卫生出版社）
"""

    print("\n=== 强约束校验结果 ===")
    validation_result = constraint_system.validate_answer(test_answer, valid_doc_ids)
    print(validation_result)

=== 带强约束的完整提示词 ===
【知识库边界规则】
你只能根据提供的参考资料回答问题，严禁编造内容。
如果问题无法从参考资料中找到答案，请明确回答：「根据现有文献无法回答此问题」。

【引用来源规则】
1. 回答中涉及的每一个事实、数据、结论，都必须在句末标注对应的来源，格式为：[文献N]
2. 只能使用本次提供的文献编号作为引用，不得编造不存在的文献编号
3. 参考资料列表：
[文献1] 为本次参考资料的第1篇文档
[文献2] 为本次参考资料的第2篇文档

【禁止编造规则】
1. 严禁添加参考资料中未提及的数据、结论、百分比或细节
2. 严禁使用「研究表明」「已被证明」「100%有效」等无依据表述
3. 回答中的所有信息必须能在参考资料中找到原文对应内容

【输出格式与术语规范】
1. 医学术语缩写首次出现时，必须给出全称，例如：β受体阻滞剂（mi类药物）
2. 回答必须包含以下固定章节标题：
   - 核心答案
   - 证据总结
   - 参考文献
3. 参考文献部分需列出本次引用的所有文献，格式为：[文献N] 文档标题（年份，来源）


用户问题：mi对心血管疾病有何影响？
参考资料：
[文献1] 心血管疾病药物治疗指南
mi类药物可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，长期使用需监测血钾。
[文献2] 临床药理学手册
mi类药物具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状。

=== 强约束校验结果 ===
{'citation_check': {'has_invalid_citation': False, 'invalid_citations': [], 'missing_citations': [], 'is_citation_valid': True}, 'format_check': {'missing_sections': [], 'abbr_errors': [], 'is_format_valid': True}, 'is_all_valid': True}


In [15]:
from typing import List, Dict, Any
import pandas as pd

# 1. 对抗测试用例生成器
class AdversarialTestGenerator:
    def __init__(self, knowledge_base_docs: List[Dict[str, Any]]):
        self.knowledge_base_docs = knowledge_base_docs
        self.test_cases = []

    def create_out_of_scope_questions(self) -> List[Dict[str, Any]]:
        """超出知识库边界的问题，期望回答：无法回答"""
        return [
            {
                "category": "out_of_scope",
                "question": "2025年mi类药物的最新治疗方法是什么？",
                "expected_response": "无法回答",
                "context_chunks": []  # 无相关上下文
            },
            {
                "category": "out_of_scope",
                "question": "mi类药物的长期致癌风险数据有哪些？",
                "expected_response": "无法回答",
                "context_chunks": []
            },
            {
                "category": "out_of_scope",
                "question": "不同种族人群使用mi类药物的剂量差异是什么？",
                "expected_response": "无法回答",
                "context_chunks": []
            }
        ]

    def create_fabrication_inducing_questions(self) -> List[Dict[str, Any]]:
        """诱导编造数据的问题，期望模型拒绝编造"""
        return [
            {
                "category": "fabrication_inducing",
                "question": "mi类药物除了降低心率外，还有哪些未提及的副作用？",
                "expected_response": "无法回答/无相关信息",
                "context_chunks": self.knowledge_base_docs
            },
            {
                "category": "fabrication_inducing",
                "question": "mi类药物的临床治愈率是多少？",
                "expected_response": "无法回答/无相关信息",
                "context_chunks": self.knowledge_base_docs
            },
            {
                "category": "fabrication_inducing",
                "question": "请补充mi类药物在糖尿病患者中的额外注意事项",
                "expected_response": "无法回答/无相关信息",
                "context_chunks": self.knowledge_base_docs
            }
        ]

    def create_term_explanation_questions(self) -> List[Dict[str, Any]]:
        """术语解释问题，检查首次出现是否带全称"""
        return [
            {
                "category": "term_explanation",
                "question": "mi类药物是什么？",
                "expected_response": "必须给出全称，如β受体阻滞剂（mi类药物）",
                "context_chunks": self.knowledge_base_docs
            },
            {
                "category": "term_explanation",
                "question": "请解释ACEI在心血管疾病中的作用",
                "expected_response": "必须给出全称，如血管紧张素转换酶抑制剂（ACEI）",
                "context_chunks": []  # 无上下文，期望回答无法解释
            }
        ]

    def generate_all_test_cases(self) -> List[Dict[str, Any]]:
        """生成全部对抗测试用例"""
        self.test_cases = (
            self.create_out_of_scope_questions() +
            self.create_fabrication_inducing_questions() +
            self.create_term_explanation_questions()
        )
        return self.test_cases

# 2. 对抗测试结果验证与指标统计器
class AdversarialTestEvaluator:
    def __init__(self, constraint_system: StrongConstraintSystem):
        self.constraint_system = constraint_system

    def _check_cannot_answer(self, answer: str) -> bool:
        """检查是否正确输出无法回答"""
        return "无法回答" in answer or "无相关信息" in answer

    def evaluate_single_case(self, case: Dict[str, Any], answer: str) -> Dict[str, Any]:
        """单个测试用例评估"""
        category = case["category"]
        eval_result = {
            "question": case["question"],
            "category": category,
            "answer": answer,
            "is_pass": False
        }

        if category == "out_of_scope" or category == "fabrication_inducing":
            # 超出范围/诱导编造：检查是否输出无法回答
            eval_result["is_pass"] = self._check_cannot_answer(answer)
            eval_result["reason"] = "模型正确拒绝回答" if eval_result["is_pass"] else "模型编造了答案"

        elif category == "term_explanation":
            # 术语解释：检查缩写是否带全称
            validation = self.constraint_system.validate_format(answer)
            eval_result["abbr_errors"] = validation["abbr_errors"]
            eval_result["is_pass"] = len(validation["abbr_errors"]) == 0
            eval_result["reason"] = "术语格式正确" if eval_result["is_pass"] else "术语缩写未带全称"

        # 额外检查引用与格式
        citation_check = self.constraint_system.validate_citations(answer, [1,2])
        format_check = self.constraint_system.validate_format(answer)
        eval_result["citation_valid"] = citation_check["is_citation_valid"]
        eval_result["format_valid"] = format_check["is_format_valid"]
        eval_result["is_all_constraint_pass"] = citation_check["is_citation_valid"] and format_check["is_format_valid"]

        return eval_result

    def calculate_overall_metrics(self, results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """统计幻觉率、引用准确率、格式合规率"""
        total = len(results)
        pass_count = sum(1 for r in results if r["is_pass"])
        hallucination_count = sum(1 for r in results if not r["is_pass"])
        citation_pass = sum(1 for r in results if r["citation_valid"])
        format_pass = sum(1 for r in results if r["format_valid"])

        return {
            "total_test_cases": total,
            "pass_rate": round(pass_count / total, 2),
            "hallucination_rate": round(hallucination_count / total, 2),
            "citation_accuracy": round(citation_pass / total, 2),
            "format_compliance_rate": round(format_pass / total, 2)
        }

    def generate_report(self, results: List[Dict[str, Any]]) -> pd.DataFrame:
        """生成测试报告DataFrame"""
        return pd.DataFrame([{
            "category": r["category"],
            "question": r["question"],
            "answer": r["answer"][:100] + "..." if len(r["answer"]) > 100 else r["answer"],
            "is_pass": r["is_pass"],
            "reason": r["reason"],
            "citation_valid": r["citation_valid"],
            "format_valid": r["format_valid"]
        } for r in results])

# 3. 测试运行（与现有流程集成）
if __name__ == "__main__":
    # 初始化知识库文档（沿用之前的测试数据）
    test_chunks = [
        {"doc_id": 1, "title": "心血管疾病药物治疗指南", "content": "mi类药物可降低心率与心肌耗氧量，常用于稳定型心绞痛的治疗，长期使用需监测血钾。", "year": 2023, "source": "临床药学杂志"},
        {"doc_id": 2, "title": "临床药理学手册", "content": "mi类药物具有轻度血管扩张作用，可改善外周循环，缓解部分患者症状。", "year": 2022, "source": "人民卫生出版社"}
    ]

    # 1. 生成对抗测试用例
    test_generator = AdversarialTestGenerator(test_chunks)
    test_cases = test_generator.generate_all_test_cases()
    print("=== 生成的对抗测试用例 ===")
    for case in test_cases:
        print(f"[{case['category']}] {case['question']}")

    # 2. 初始化强约束校验器
    constraint_system = StrongConstraintSystem()
    test_evaluator = AdversarialTestEvaluator(constraint_system)

    # 3. 模拟模型回答并评估
    simulated_answers = [
        "根据现有文献无法回答此问题。",  # 超出知识库
        "根据现有文献无法回答此问题。",  # 超出知识库
        "根据现有文献无法回答此问题。",  # 超出知识库
        "根据现有文献无法回答此问题。",  # 诱导编造
        "根据现有文献无法回答此问题。",  # 诱导编造
        "根据现有文献无法回答此问题。",  # 诱导编造
        "β受体阻滞剂（mi类药物）是一类可降低心率与心肌耗氧量的药物，常用于稳定型心绞痛的治疗[文献1][文献2]。",  # 术语解释
        "根据现有文献无法回答此问题。"  # 无上下文术语
    ]

    test_results = []
    for case, answer in zip(test_cases, simulated_answers):
        res = test_evaluator.evaluate_single_case(case, answer)
        test_results.append(res)

    # 4. 输出指标与报告
    metrics = test_evaluator.calculate_overall_metrics(test_results)
    print("\n=== 对抗测试指标统计 ===")
    print(metrics)

    print("\n=== 测试结果报告 ===")
    report_df = test_evaluator.generate_report(test_results)
    print(report_df)

=== 生成的对抗测试用例 ===
[out_of_scope] 2025年mi类药物的最新治疗方法是什么？
[out_of_scope] mi类药物的长期致癌风险数据有哪些？
[out_of_scope] 不同种族人群使用mi类药物的剂量差异是什么？
[fabrication_inducing] mi类药物除了降低心率外，还有哪些未提及的副作用？
[fabrication_inducing] mi类药物的临床治愈率是多少？
[fabrication_inducing] 请补充mi类药物在糖尿病患者中的额外注意事项
[term_explanation] mi类药物是什么？
[term_explanation] 请解释ACEI在心血管疾病中的作用

=== 对抗测试指标统计 ===
{'total_test_cases': 8, 'pass_rate': 1.0, 'hallucination_rate': 0.0, 'citation_accuracy': 1.0, 'format_compliance_rate': 0.0}

=== 测试结果报告 ===
               category                   question  \
0          out_of_scope      2025年mi类药物的最新治疗方法是什么？   
1          out_of_scope         mi类药物的长期致癌风险数据有哪些？   
2          out_of_scope     不同种族人群使用mi类药物的剂量差异是什么？   
3  fabrication_inducing  mi类药物除了降低心率外，还有哪些未提及的副作用？   
4  fabrication_inducing            mi类药物的临床治愈率是多少？   
5  fabrication_inducing     请补充mi类药物在糖尿病患者中的额外注意事项   
6      term_explanation                  mi类药物是什么？   
7      term_explanation          请解释ACEI在心血管疾病中的作用   

                          